In [1]:
!pip install -q pandas requests python-dotenv


In [2]:
import pandas as pd
import requests
import os

print("Libraries imported successfully!")


Libraries imported successfully!


In [3]:
jobs = [
    {
        "title": "Data Analyst Intern",
        "company": "ABC Technologies",
        "location": "Remote",
        "skills": ["Python", "SQL", "Pandas", "Excel"],
        "experience": "Fresher"
    },
    {
        "title": "Machine Learning Intern",
        "company": "AI Solutions",
        "location": "Noida",
        "skills": ["Python", "Machine Learning", "Scikit-learn"],
        "experience": "Fresher"
    },
    {
        "title": "Python Developer Intern",
        "company": "Tech Labs",
        "location": "Delhi",
        "skills": ["Python", "Django", "SQL"],
        "experience": "Fresher"
    },
    {
        "title": "Data Science Intern",
        "company": "DataWorks",
        "location": "Remote",
        "skills": ["Python", "Machine Learning", "Pandas", "NumPy"],
        "experience": "Fresher"
    }
]

print("Number of jobs:", len(jobs))

Number of jobs: 4


In [4]:
def search_jobs(user_skills):
    results = []

    for job in jobs:
        matched_skills = []

        for skill in user_skills:
            if skill.lower() in [s.lower() for s in job["skills"]]:
                matched_skills.append(skill)

        if matched_skills:
            results.append({
                "title": job["title"],
                "company": job["company"],
                "location": job["location"],
                "matched_skills": matched_skills,
                "all_skills": job["skills"]
            })

    return results


print("Job search tool created!")

Job search tool created!


In [5]:
user_skills = ["Python", "SQL"]

results = search_jobs(user_skills)

for job in results:
    print(job)

{'title': 'Data Analyst Intern', 'company': 'ABC Technologies', 'location': 'Remote', 'matched_skills': ['Python', 'SQL'], 'all_skills': ['Python', 'SQL', 'Pandas', 'Excel']}
{'title': 'Machine Learning Intern', 'company': 'AI Solutions', 'location': 'Noida', 'matched_skills': ['Python'], 'all_skills': ['Python', 'Machine Learning', 'Scikit-learn']}
{'title': 'Python Developer Intern', 'company': 'Tech Labs', 'location': 'Delhi', 'matched_skills': ['Python', 'SQL'], 'all_skills': ['Python', 'Django', 'SQL']}
{'title': 'Data Science Intern', 'company': 'DataWorks', 'location': 'Remote', 'matched_skills': ['Python'], 'all_skills': ['Python', 'Machine Learning', 'Pandas', 'NumPy']}


In [7]:
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")

print("API key loaded successfully!")

API key loaded successfully!


In [8]:
!pip install -q openai

In [9]:
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

print("AI model connected successfully!")

AI model connected successfully!


In [11]:
def ask_ai(user_request):
    response = client.responses.create(
        model="gpt-6-luna",
        input=f"""
You are an AI Job Finder Agent.

The user wants:
{user_request}

Available jobs:
{jobs}

Analyze the available jobs and recommend the most relevant ones.
For each recommended job, explain why it matches the user's request.
"""
    )

    return response.output_text


result = ask_ai(
    "I am a fresher looking for a Data Analyst internship. "
    "I know Python, SQL and Pandas."
)

print(result)

### Recommended jobs

1. **Data Analyst Intern — ABC Technologies (Remote)**  
   **Best match:** The role directly matches your goal, and you already know Python, SQL, and Pandas—all listed skills. Excel is the only additional skill listed.

2. **Data Science Intern — DataWorks (Remote)**  
   **Related option:** It’s a fresher-level, remote internship that uses Python and Pandas. It also emphasizes Machine Learning and NumPy, which aren’t among your stated skills, so it’s a less direct match than the Data Analyst role.


In [12]:
import json

def search_jobs_tool(user_skills):
    results = search_jobs(user_skills)

    return json.dumps(results, indent=2)


print(search_jobs_tool(["Python", "SQL"]))

[
  {
    "title": "Data Analyst Intern",
    "company": "ABC Technologies",
    "location": "Remote",
    "matched_skills": [
      "Python",
      "SQL"
    ],
    "all_skills": [
      "Python",
      "SQL",
      "Pandas",
      "Excel"
    ]
  },
  {
    "title": "Machine Learning Intern",
    "company": "AI Solutions",
    "location": "Noida",
    "matched_skills": [
      "Python"
    ],
    "all_skills": [
      "Python",
      "Machine Learning",
      "Scikit-learn"
    ]
  },
  {
    "title": "Python Developer Intern",
    "company": "Tech Labs",
    "location": "Delhi",
    "matched_skills": [
      "Python",
      "SQL"
    ],
    "all_skills": [
      "Python",
      "Django",
      "SQL"
    ]
  },
  {
    "title": "Data Science Intern",
    "company": "DataWorks",
    "location": "Remote",
    "matched_skills": [
      "Python"
    ],
    "all_skills": [
      "Python",
      "Machine Learning",
      "Pandas",
      "NumPy"
    ]
  }
]


In [13]:
tools = [
    {
        "type": "function",
        "name": "search_jobs",
        "description": "Search available jobs based on the skills provided by the user.",
        "parameters": {
            "type": "object",
            "properties": {
                "user_skills": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    },
                    "description": "Skills the user has, such as Python, SQL, Pandas."
                }
            },
            "required": ["user_skills"],
            "additionalProperties": False
        }
    }
]

print("Tool definition created!")

Tool definition created!


In [14]:
import json

def run_agent(user_request):

    # Step 1: Ask the AI what to do
    response = client.responses.create(
        model="gpt-6-luna",
        instructions="""
        You are an AI Job Finder Agent.

        Your job is to help users find relevant jobs.
        You have access to a search_jobs tool.

        If you need job information, use the tool.
        After receiving the tool results, recommend the
        most relevant jobs to the user.
        """,
        input=user_request,
        tools=tools
    )

    # Step 2: Check if the AI wants to use a tool
    for item in response.output:

        if item.type == "function_call":

            if item.name == "search_jobs":

                # Step 3: Get the skills chosen by the AI
                arguments = json.loads(item.arguments)

                user_skills = arguments["user_skills"]

                # Step 4: Run our Python function
                job_results = search_jobs(user_skills)

                # Step 5: Send the tool result back to the AI
                response = client.responses.create(
                    model="gpt-6-luna",
                    instructions="""
                    You are an AI Job Finder Agent.
                    Analyze the job results and recommend
                    the most relevant jobs to the user.
                    Explain why each job matches.
                    """,
                    input=[
                        {
                            "role": "user",
                            "content": user_request
                        },
                        {
                            "role": "user",
                            "content": f"Job search results: {job_results}"
                        }
                    ]
                )

                return response.output_text

    return response.output_text


print("Agent created successfully!")

Agent created successfully!


In [15]:
user_request = """
I am a fresher looking for a Data Analyst or Data Science internship.
I know Python, SQL, Pandas and Machine Learning.
I prefer Remote or Delhi/Noida jobs.
"""

result = run_agent(user_request)

print(result)

### Best matches

1. **Data Science Intern — DataWorks**  
   **Remote** · Matches **Python, Pandas, and Machine Learning**. This is an exact match for your Data Science internship goal and preferred location.

2. **Data Analyst Intern — ABC Technologies**  
   **Remote** · Matches **Python, SQL, and Pandas**. A strong fit for your Data Analyst goal and your remote-work preference. **Excel** is also listed as a required skill, so consider brushing up if you haven’t used it.

3. **Machine Learning Intern — AI Solutions**  
   **Noida** · Matches **Python and Machine Learning**. It’s relevant to your Data Science interests and fits your location preference. **Scikit-learn** is listed as an additional skill.

### Also worth considering

4. **Python Developer Intern — Tech Labs**  
   **Delhi** · Matches **Python and SQL**. The location and internship level fit, but the role is less focused on data analysis or data science than the others.


In [16]:
def calculate_match_score(user_skills, job_skills):
    user_skills_lower = [skill.lower() for skill in user_skills]
    job_skills_lower = [skill.lower() for skill in job_skills]

    matched = set(user_skills_lower) & set(job_skills_lower)

    if len(job_skills_lower) == 0:
        return 0

    score = (len(matched) / len(job_skills_lower)) * 100

    return round(score, 2)


print(calculate_match_score(
    ["Python", "SQL", "Pandas"],
    ["Python", "SQL", "Pandas", "Excel"]
))

75.0


In [17]:
def search_jobs(user_skills):
    results = []

    for job in jobs:

        matched_skills = []

        for skill in user_skills:
            if skill.lower() in [s.lower() for s in job["skills"]]:
                matched_skills.append(skill)

        score = calculate_match_score(
            user_skills,
            job["skills"]
        )

        if matched_skills:
            results.append({
                "title": job["title"],
                "company": job["company"],
                "location": job["location"],
                "matched_skills": matched_skills,
                "all_skills": job["skills"],
                "match_score": score
            })

    results.sort(
        key=lambda x: x["match_score"],
        reverse=True
    )

    return results


print(search_jobs(["Python", "SQL", "Pandas"]))

[{'title': 'Data Analyst Intern', 'company': 'ABC Technologies', 'location': 'Remote', 'matched_skills': ['Python', 'SQL', 'Pandas'], 'all_skills': ['Python', 'SQL', 'Pandas', 'Excel'], 'match_score': 75.0}, {'title': 'Python Developer Intern', 'company': 'Tech Labs', 'location': 'Delhi', 'matched_skills': ['Python', 'SQL'], 'all_skills': ['Python', 'Django', 'SQL'], 'match_score': 66.67}, {'title': 'Data Science Intern', 'company': 'DataWorks', 'location': 'Remote', 'matched_skills': ['Python', 'Pandas'], 'all_skills': ['Python', 'Machine Learning', 'Pandas', 'NumPy'], 'match_score': 50.0}, {'title': 'Machine Learning Intern', 'company': 'AI Solutions', 'location': 'Noida', 'matched_skills': ['Python'], 'all_skills': ['Python', 'Machine Learning', 'Scikit-learn'], 'match_score': 33.33}]


In [18]:
import ipywidgets as widgets
from IPython.display import display, clear_output

role_input = widgets.Text(
    description="Job Role:",
    placeholder="e.g. Data Analyst Intern"
)

skills_input = widgets.Text(
    description="Skills:",
    placeholder="e.g. Python, SQL, Pandas"
)

location_input = widgets.Text(
    description="Location:",
    placeholder="e.g. Delhi, Noida, Remote"
)

search_button = widgets.Button(
    description="🔎 Find Jobs"
)

output = widgets.Output()


def on_search_clicked(b):

    with output:
        clear_output()

        role = role_input.value
        skills = [
            skill.strip()
            for skill in skills_input.value.split(",")
            if skill.strip()
        ]
        location = location_input.value

        request = f"""
        Find jobs for me.

        Job role: {role}
        Skills: {skills}
        Preferred location: {location}

        Recommend the most relevant jobs.
        Explain why each job matches.
        """

        print("🤖 Agent is searching...\n")

        result = run_agent(request)

        print(result)


search_button.on_click(on_search_clicked)

display(
    role_input,
    skills_input,
    location_input,
    search_button,
    output
)

Text(value='', description='Job Role:', placeholder='e.g. Data Analyst Intern')

Text(value='', description='Skills:', placeholder='e.g. Python, SQL, Pandas')

Text(value='', description='Location:', placeholder='e.g. Delhi, Noida, Remote')

Button(description='🔎 Find Jobs', style=ButtonStyle())

Output()

In [19]:
def search_jobs(user_skills, role=None, location=None):
    results = []

    for job in jobs:

        # Skill matching
        matched_skills = []

        for skill in user_skills:
            if skill.lower() in [s.lower() for s in job["skills"]]:
                matched_skills.append(skill)

        # Role matching
        role_match = True

        if role:
            role_words = role.lower().split()
            job_title = job["title"].lower()

            role_match = any(
                word in job_title
                for word in role_words
                if len(word) > 2
            )

        # Location matching
        location_match = True

        if location:
            preferred_locations = [
                x.strip().lower()
                for x in location.split(",")
            ]

            location_match = (
                job["location"].lower() in preferred_locations
                or "remote" in preferred_locations
                and job["location"].lower() == "remote"
            )

        # Calculate score
        score = calculate_match_score(
            user_skills,
            job["skills"]
        )

        if matched_skills and role_match and location_match:

            results.append({
                "title": job["title"],
                "company": job["company"],
                "location": job["location"],
                "matched_skills": matched_skills,
                "all_skills": job["skills"],
                "match_score": score
            })

    results.sort(
        key=lambda x: x["match_score"],
        reverse=True
    )

    return results

In [20]:
results = search_jobs(
    ["Python", "SQL", "Pandas"],
    role="Data Analyst Intern",
    location="Remote"
)

print(results)

[{'title': 'Data Analyst Intern', 'company': 'ABC Technologies', 'location': 'Remote', 'matched_skills': ['Python', 'SQL', 'Pandas'], 'all_skills': ['Python', 'SQL', 'Pandas', 'Excel'], 'match_score': 75.0}, {'title': 'Data Science Intern', 'company': 'DataWorks', 'location': 'Remote', 'matched_skills': ['Python', 'Pandas'], 'all_skills': ['Python', 'Machine Learning', 'Pandas', 'NumPy'], 'match_score': 50.0}]


In [21]:
tools = [
    {
        "type": "function",
        "name": "search_jobs",
        "description": "Search jobs based on user's skills, job role, and preferred location.",
        "parameters": {
            "type": "object",
            "properties": {
                "user_skills": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Skills the user has"
                },
                "role": {
                    "type": "string",
                    "description": "Job role the user wants"
                },
                "location": {
                    "type": "string",
                    "description": "Preferred job location"
                }
            },
            "required": ["user_skills", "role", "location"],
            "additionalProperties": False
        }
    }
]

print("Updated tool created!")

Updated tool created!


In [22]:
import json

def run_agent(user_request):

    response = client.responses.create(
        model="gpt-6-luna",
        instructions="""
        You are an AI Job Finder Agent.

        Your job is to help users find relevant jobs.

        Extract these from the user's request:
        1. Job role
        2. Skills
        3. Preferred location

        Use the search_jobs tool to find matching jobs.

        After receiving the results, recommend the most relevant jobs.
        Explain the match score and why each job is relevant.
        """,
        input=user_request,
        tools=tools
    )

    for item in response.output:

        if item.type == "function_call":

            if item.name == "search_jobs":

                arguments = json.loads(item.arguments)

                user_skills = arguments["user_skills"]
                role = arguments["role"]
                location = arguments["location"]

                job_results = search_jobs(
                    user_skills,
                    role=role,
                    location=location
                )

                final_response = client.responses.create(
                    model="gpt-6-luna",
                    instructions="""
                    You are an AI Job Finder Agent.

                    Analyze the job search results.

                    For each job:
                    - Give the job title
                    - Company
                    - Location
                    - Match score
                    - Matched skills
                    - Briefly explain why it matches

                    If no jobs are found, clearly say that no matching
                    jobs were found.
                    """,
                    input=f"""
                    User request:
                    {user_request}

                    Job search results:
                    {job_results}
                    """
                )

                return final_response.output_text

    return response.output_text


print("Agent updated successfully!")

Agent updated successfully!


In [26]:
def search_jobs(user_skills, role=None, location=None):
    results = []

    for job in jobs:

        # Skill matching
        matched_skills = []

        for skill in user_skills:
            if skill.lower() in [s.lower() for s in job["skills"]]:
                matched_skills.append(skill)

        # Role matching
        role_match = True

        if role:
            role_words = role.lower().split()
            job_title = job["title"].lower()

            role_match = any(
                word in job_title
                for word in role_words
                if len(word) > 2
            )

        # Location matching
        location_match = True

        if location:
            preferred_locations = [
                x.strip().lower()
                for x in location.split(",")
                if x.strip()
            ]

            job_location = job["location"].lower()

            location_match = (
                job_location in preferred_locations
                or job_location == "remote"
            )

        # Match score
        score = calculate_match_score(
            user_skills,
            job["skills"]
        )

        if matched_skills and role_match and location_match:

            results.append({
                "title": job["title"],
                "company": job["company"],
                "location": job["location"],
                "matched_skills": matched_skills,
                "all_skills": job["skills"],
                "match_score": score
            })

    results.sort(
        key=lambda x: x["match_score"],
        reverse=True
    )

    return results


print("Search function updated!")

Search function updated!


In [28]:
print(search_jobs(
    ["Python", "Machine Learning"],
    role="Data Science Intern",
    location="Noida"
))

[{'title': 'Machine Learning Intern', 'company': 'AI Solutions', 'location': 'Noida', 'matched_skills': ['Python', 'Machine Learning'], 'all_skills': ['Python', 'Machine Learning', 'Scikit-learn'], 'match_score': 66.67}, {'title': 'Data Science Intern', 'company': 'DataWorks', 'location': 'Remote', 'matched_skills': ['Python', 'Machine Learning'], 'all_skills': ['Python', 'Machine Learning', 'Pandas', 'NumPy'], 'match_score': 50.0}, {'title': 'Data Analyst Intern', 'company': 'ABC Technologies', 'location': 'Remote', 'matched_skills': ['Python'], 'all_skills': ['Python', 'SQL', 'Pandas', 'Excel'], 'match_score': 25.0}]


In [27]:
user_request = """
I am a fresher looking for a Data Analyst internship.
I know Python, SQL and Pandas.
I prefer Remote jobs.
"""

result = run_agent(user_request)

print(result)

### 1. Data Analyst Intern
- **Company:** ABC Technologies
- **Location:** Remote
- **Match score:** 75%
- **Matched skills:** Python, SQL, Pandas
- **Why it matches:** It’s a remote data analyst internship and matches all three of your listed skills.

### 2. Data Science Intern
- **Company:** DataWorks
- **Location:** Remote
- **Match score:** 50%
- **Matched skills:** Python, Pandas
- **Why it matches:** It’s a remote internship and matches Python and Pandas, though it’s a data science role rather than a data analyst role and doesn’t list SQL.


In [24]:
!pip install -q gradio

In [31]:
def search_jobs(user_skills, role=None, location=None):

    results = []

    for job in jobs:

        # Skills
        matched_skills = [
            skill for skill in user_skills
            if skill.lower() in [s.lower() for s in job["skills"]]
        ]

        if not matched_skills:
            continue

        # Role matching
        role_match = True

        if role:
            role_words = [
                word.lower()
                for word in role.split()
                if len(word) > 2
            ]

            job_title = job["title"].lower()

            role_match = any(
                word in job_title
                for word in role_words
            )

        if not role_match:
            continue

        # Location
        location_match = True

        if location:
            preferred = [
                x.strip().lower()
                for x in location.split(",")
                if x.strip()
            ]

            job_location = job["location"].lower()

            # Exact location OR Remote
            location_match = (
                job_location in preferred
                or job_location == "remote"
            )

        if not location_match:
            continue

        score = calculate_match_score(
            user_skills,
            job["skills"]
        )

        results.append({
            "title": job["title"],
            "company": job["company"],
            "location": job["location"],
            "matched_skills": matched_skills,
            "all_skills": job["skills"],
            "match_score": score
        })

    results.sort(
        key=lambda x: x["match_score"],
        reverse=True
    )

    return results


print("Search function fixed!")

Search function fixed!


In [32]:
print(
    search_jobs(
        ["Python", "Machine Learning"],
        role="Data Science Intern",
        location="Noida"
    )
)

[{'title': 'Machine Learning Intern', 'company': 'AI Solutions', 'location': 'Noida', 'matched_skills': ['Python', 'Machine Learning'], 'all_skills': ['Python', 'Machine Learning', 'Scikit-learn'], 'match_score': 66.67}, {'title': 'Data Science Intern', 'company': 'DataWorks', 'location': 'Remote', 'matched_skills': ['Python', 'Machine Learning'], 'all_skills': ['Python', 'Machine Learning', 'Pandas', 'NumPy'], 'match_score': 50.0}, {'title': 'Data Analyst Intern', 'company': 'ABC Technologies', 'location': 'Remote', 'matched_skills': ['Python'], 'all_skills': ['Python', 'SQL', 'Pandas', 'Excel'], 'match_score': 25.0}]


In [33]:
import json

def run_agent(user_request):

    response = client.responses.create(
        model="gpt-6-luna",
        instructions="""
        You are an AI Job Finder Agent.

        Read the user's request and extract:
        - job role
        - skills
        - preferred location

        You MUST use the search_jobs tool to find jobs.
        """,
        input=user_request,
        tools=tools
    )

    for item in response.output:

        if item.type == "function_call" and item.name == "search_jobs":

            arguments = json.loads(item.arguments)

            user_skills = arguments["user_skills"]
            role = arguments["role"]
            location = arguments["location"]

            job_results = search_jobs(
                user_skills=user_skills,
                role=role,
                location=location
            )

            if not job_results:
                return """
### 😕 No exact matching jobs found

Try:
- adding more skills
- selecting Remote
- trying Delhi / Noida
- using a broader job role
"""

            final_prompt = f"""
            User request:
            {user_request}

            Matching jobs:
            {job_results}

            Recommend the matching jobs clearly.

            For each job show:
            - Job title
            - Company
            - Location
            - Match score
            - Matched skills
            - Why it matches
            """

            final_response = client.responses.create(
                model="gpt-6-luna",
                input=final_prompt
            )

            return final_response.output_text

    return response.output_text


print("Agent fixed successfully!")

Agent fixed successfully!


In [34]:
print(run_agent("""
I am a fresher looking for a Data Science Intern role.
I know Python and Machine Learning.
I prefer Noida.
"""))

## Recommended matching jobs

### 1. Machine Learning Intern — AI Solutions
- **Location:** Noida
- **Match score:** 66.67%
- **Matched skills:** Python, Machine Learning
- **Why it matches:** It matches both of your stated skills and is in your preferred location. The role is an internship, though its focus is Machine Learning rather than Data Science.

### 2. Data Science Intern — DataWorks
- **Location:** Remote
- **Match score:** 50.0%
- **Matched skills:** Python, Machine Learning
- **Why it matches:** The job title directly aligns with your target role, and it matches both of your stated skills. It’s remote rather than based in Noida.

### 3. Data Analyst Intern — ABC Technologies
- **Location:** Remote
- **Match score:** 25.0%
- **Matched skills:** Python
- **Why it matches:** This internship matches your Python skill, but not Machine Learning. It’s a less direct fit for your Data Science goal and is remote rather than in Noida.

**Best overall fit:** AI Solutions, for its Noida

In [36]:
def find_jobs(role, skills, location):

    skill_list = [
        skill.strip()
        for skill in skills.split(",")
        if skill.strip()
    ]

    # Directly search our job database
    results = search_jobs(
        user_skills=skill_list,
        role=role,
        location=location
    )

    if not results:
        return """
### 😕 No matching jobs found

Try adding more skills or selecting Remote.
"""

    # Let AI explain the results
    prompt = f"""
You are an AI Job Finder.

User:
Job Role: {role}
Skills: {skill_list}
Preferred Location: {location}

Matching jobs:
{results}

Show the results in a clean format.

For each job include:
- 💼 Job Title
- 🏢 Company
- 📍 Location
- ⭐ Match Score
- 🛠️ Matched Skills
- 💡 Why it matches
"""

    response = client.responses.create(
        model="gpt-6-luna",
        input=prompt
    )

    return response.output_text

In [38]:
test_results = search_jobs(
    user_skills=["Python", "Machine Learning"],
    role="Data Science Intern",
    location="Noida"
)

print(test_results)

[{'title': 'Machine Learning Intern', 'company': 'AI Solutions', 'location': 'Noida', 'matched_skills': ['Python', 'Machine Learning'], 'all_skills': ['Python', 'Machine Learning', 'Scikit-learn'], 'match_score': 66.67}, {'title': 'Data Science Intern', 'company': 'DataWorks', 'location': 'Remote', 'matched_skills': ['Python', 'Machine Learning'], 'all_skills': ['Python', 'Machine Learning', 'Pandas', 'NumPy'], 'match_score': 50.0}, {'title': 'Data Analyst Intern', 'company': 'ABC Technologies', 'location': 'Remote', 'matched_skills': ['Python'], 'all_skills': ['Python', 'SQL', 'Pandas', 'Excel'], 'match_score': 25.0}]


In [40]:
test = find_jobs(
    "Data Science Intern",
    "Python, Machine Learning",
    "Noida"
)

print(test)

### 1. 💼 Machine Learning Intern
- 🏢 **Company:** AI Solutions
- 📍 **Location:** Noida
- ⭐ **Match Score:** 66.67%
- 🛠️ **Matched Skills:** Python, Machine Learning
- 💡 **Why it matches:** It matches both of your listed skills and is located in your preferred area.

### 2. 💼 Data Science Intern
- 🏢 **Company:** DataWorks
- 📍 **Location:** Remote
- ⭐ **Match Score:** 50%
- 🛠️ **Matched Skills:** Python, Machine Learning
- 💡 **Why it matches:** The job title matches your preferred role, and it includes both of your listed skills. It’s remote rather than in Noida.

### 3. 💼 Data Analyst Intern
- 🏢 **Company:** ABC Technologies
- 📍 **Location:** Remote
- ⭐ **Match Score:** 25%
- 🛠️ **Matched Skills:** Python
- 💡 **Why it matches:** It includes Python, one of your listed skills. It’s a different role and is remote rather than in Noida.


In [42]:
import gradio as gr

demo = gr.Interface(
    fn=find_jobs,
    inputs=[
        gr.Textbox(
            label="💼 Job Role",
            placeholder="Data Science Intern"
        ),
        gr.Textbox(
            label="🛠️ Skills",
            placeholder="Python, Machine Learning"
        ),
        gr.Textbox(
            label="📍 Preferred Location",
            placeholder="Noida"
        )
    ],
    outputs=gr.Markdown(
        label="🤖 AI Job Recommendations"
    ),
    title="🤖 AI Job Finder Agent",
    description="Enter your job preferences and get AI-powered job recommendations.",
    submit_btn="🔍 Find Jobs"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3249eeb06a6101d9db.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [44]:
demo.launch(share=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3249eeb06a6101d9db.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [51]:
import gradio as gr

location_choices = [
    "Delhi",
    "Noida",
    "Mumbai",
    "Banaras",
    "Bangalore",
    "Kolkata",
    "Pune",
    "Nagpur",
    "Hyderabad",
    "Chennai",
    "Ahmedabad",
    "Jaipur",
    "Gurgaon",
    "Remote"
]


def job_finder_ui(role, skills, locations):

    skill_list = [
        s.strip()
        for s in skills.split(",")
        if s.strip()
    ]

    # Multiple selected locations
    location = ", ".join(locations)

    results = search_jobs(
        user_skills=skill_list,
        role=role,
        location=location
    )

    if not results:
        return "❌ No matching jobs found for the selected locations."

    output = "## 🤖 Recommended Jobs\n\n"

    for i, job in enumerate(results, 1):

        output += f"""
### {i}. 💼 {job['title']}

**🏢 Company:** {job['company']}
**📍 Location:** {job['location']}
**⭐ Match Score:** {job['match_score']}%
**🛠️ Matched Skills:** {", ".join(job['matched_skills'])}

---

"""

    return output


demo = gr.Interface(
    fn=job_finder_ui,

    inputs=[
        gr.Textbox(
            label="💼 Job Role",
            placeholder="e.g. Data Science Intern"
        ),

        gr.Textbox(
            label="🛠️ Skills",
            placeholder="e.g. Python, ML, SQL"
        ),

        gr.CheckboxGroup(
            choices=location_choices,
            label="📍 Preferred Locations",
            info="Select one or multiple locations"
        )
    ],

    outputs=gr.Markdown(),

    title="🤖 AI Job Finder Agent",

    description="Find internship opportunities based on your role, skills and preferred locations.",

    submit_btn="🔍 Find Jobs"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9531682a90d02cbeed.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [47]:
skill_aliases = {
    "ml": "machine learning",
    "machine learning": "machine learning",
    "python": "python",
    "sql": "sql",
    "pandas": "pandas",
    "numpy": "numpy",
    "sklearn": "scikit-learn",
    "scikit learn": "scikit-learn"
}


def normalize_skill(skill):
    skill = skill.strip().lower()
    return skill_aliases.get(skill, skill)


def search_jobs(user_skills, role=None, location=None):

    results = []

    normalized_user_skills = [
        normalize_skill(skill)
        for skill in user_skills
    ]

    for job in jobs:

        job_skills = [
            normalize_skill(skill)
            for skill in job["skills"]
        ]

        matched_skills = []

        for original_skill, normalized_skill in zip(
            user_skills,
            normalized_user_skills
        ):
            if normalized_skill in job_skills:
                matched_skills.append(original_skill)

        if not matched_skills:
            continue

        # Role matching
        role_match = True

        if role:
            role_words = [
                word.lower()
                for word in role.split()
                if len(word) > 2
            ]

            job_title = job["title"].lower()

            role_match = any(
                word in job_title
                for word in role_words
            )

        if not role_match:
            continue

        # Location matching
        location_match = True

        if location:
            preferred_locations = [
                x.strip().lower()
                for x in location.split(",")
                if x.strip()
            ]

            job_location = job["location"].lower()

            location_match = (
                job_location in preferred_locations
                or job_location == "remote"
            )

        if not location_match:
            continue

        # Match score
        score = (
            len(matched_skills)
            / len(job_skills)
        ) * 100

        results.append({
            "title": job["title"],
            "company": job["company"],
            "location": job["location"],
            "matched_skills": matched_skills,
            "all_skills": job["skills"],
            "match_score": round(score, 2)
        })

    results.sort(
        key=lambda x: x["match_score"],
        reverse=True
    )

    return results


print("Skill matching improved!")

Skill matching improved!


In [48]:
print(
    search_jobs(
        ["ML"],
        role="Data Science Intern",
        location="Noida"
    )
)

[{'title': 'Machine Learning Intern', 'company': 'AI Solutions', 'location': 'Noida', 'matched_skills': ['ML'], 'all_skills': ['Python', 'Machine Learning', 'Scikit-learn'], 'match_score': 33.33}, {'title': 'Data Science Intern', 'company': 'DataWorks', 'location': 'Remote', 'matched_skills': ['ML'], 'all_skills': ['Python', 'Machine Learning', 'Pandas', 'NumPy'], 'match_score': 25.0}]


In [49]:
# Clean UI function

def job_finder_ui(role, skills, location):

    skill_list = [
        s.strip()
        for s in skills.split(",")
        if s.strip()
    ]

    results = search_jobs(
        user_skills=skill_list,
        role=role,
        location=location
    )

    if not results:
        return "❌ No matching jobs found."

    output = "## 🤖 Recommended Jobs\n\n"

    for i, job in enumerate(results, 1):

        output += f"""
### {i}. 💼 {job['title']}

**🏢 Company:** {job['company']}
**📍 Location:** {job['location']}
**⭐ Match Score:** {job['match_score']}%
**🛠️ Matched Skills:** {", ".join(job['matched_skills'])}

---
"""

    return output


print("Clean UI function ready!")

Clean UI function ready!


In [50]:
import gradio as gr

demo = gr.Interface(
    fn=job_finder_ui,

    inputs=[
        gr.Textbox(
            label="💼 Job Role",
            placeholder="Data Science Intern"
        ),

        gr.Textbox(
            label="🛠️ Skills",
            placeholder="Python, Machine Learning"
        ),

        gr.Textbox(
            label="📍 Preferred Location",
            placeholder="Noida"
        )
    ],

    outputs=gr.Markdown(),

    title="🤖 AI Job Finder Agent",

    description="""
    Find suitable internship opportunities based on your
    job role, skills and preferred location.
    """,

    submit_btn="🔍 Find Jobs"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4bb22d4dff7b715ef5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [52]:
!pip install -q pypdf python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.8 MB/s eta 0:00:00


In [53]:
from pypdf import PdfReader
from docx import Document
import os

def extract_resume_text(file_path):

    if not file_path:
        return ""

    extension = os.path.splitext(file_path)[1].lower()

    # PDF
    if extension == ".pdf":
        reader = PdfReader(file_path)

        text = ""

        for page in reader.pages:
            text += page.extract_text() or ""

        return text

    # DOCX
    elif extension == ".docx":
        document = Document(file_path)

        return "\n".join(
            paragraph.text
            for paragraph in document.paragraphs
        )

    # TXT
    elif extension == ".txt":
        with open(file_path, "r", encoding="utf-8") as f:
            return f.read()

    return ""


print("Resume reader ready!")

Resume reader ready!


In [54]:
def extract_skills_from_resume(resume_text):

    if not resume_text.strip():
        return []

    response = client.responses.create(
        model="gpt-6-luna",
        input=f"""
You are a resume skill extractor.

Read this resume and identify the technical skills.

Return ONLY a comma-separated list of skills.
Do not add explanations.

Resume:
{resume_text}
"""
    )

    skills = [
        skill.strip()
        for skill in response.output_text.split(",")
        if skill.strip()
    ]

    return skills


print("Resume skill extractor ready!")

Resume skill extractor ready!


In [55]:
def extract_skills_from_resume(resume_text):

    if not resume_text.strip():
        return []

    response = client.responses.create(
        model="gpt-6-luna",
        input=f"""
You are a resume skill extractor.

Read the resume below and identify the technical skills.

Return ONLY a comma-separated list of skills.
Do not add explanations.

Resume:
{resume_text}
"""
    )

    skills = [
        skill.strip()
        for skill in response.output_text.split(",")
        if skill.strip()
    ]

    return skills


print("Resume skill extractor ready!")

Resume skill extractor ready!


In [56]:
sample_resume = """
BTech Computer Science student.
Skills: Python, SQL, Machine Learning, Pandas, NumPy.
Experience with data analysis and scikit-learn.
"""

print(extract_skills_from_resume(sample_resume))

['Python', 'SQL', 'Machine Learning', 'Pandas', 'NumPy', 'Data Analysis', 'scikit-learn']


In [57]:
import gradio as gr

def final_job_finder(role, skills, locations, resume):

    # Get skills from resume if uploaded
    resume_skills = []

    if resume is not None:
        resume_text = extract_resume_text(resume)
        resume_skills = extract_skills_from_resume(resume_text)

    # Manual skills
    manual_skills = [
        s.strip()
        for s in skills.split(",")
        if s.strip()
    ]

    # Combine both
    all_skills = manual_skills + resume_skills

    # Remove duplicates
    all_skills = list(dict.fromkeys(
        s.lower() for s in all_skills
    ))

    if not all_skills:
        return "⚠️ Please enter skills or upload your resume."

    location_text = ", ".join(locations)

    results = search_jobs(
        user_skills=all_skills,
        role=role,
        location=location_text
    )

    if not results:
        return """
## 😕 No matching jobs found

Try adding more skills or selecting Remote.
"""

    output = "## 🤖 Recommended Jobs\n\n"

    if resume_skills:
        output += "**📄 Skills detected from resume:** "
        output += ", ".join(resume_skills)
        output += "\n\n"

    for i, job in enumerate(results, 1):

        output += f"""
### {i}. 💼 {job['title']}

**🏢 Company:** {job['company']}
**📍 Location:** {job['location']}
**⭐ Match Score:** {job['match_score']}%
**🛠️ Matched Skills:** {", ".join(job['matched_skills'])}

---

"""

    return output


locations = [
    "Delhi",
    "Noida",
    "Mumbai",
    "Banaras",
    "Bangalore",
    "Kolkata",
    "Pune",
    "Nagpur",
    "Hyderabad",
    "Chennai",
    "Ahmedabad",
    "Jaipur",
    "Gurgaon",
    "Remote"
]


app = gr.Interface(
    fn=final_job_finder,

    inputs=[
        gr.Textbox(
            label="💼 Job Role",
            placeholder="e.g. Data Science Intern"
        ),

        gr.Textbox(
            label="🛠️ Skills",
            placeholder="e.g. Python, SQL, ML"
        ),

        gr.CheckboxGroup(
            choices=locations,
            label="📍 Preferred Locations",
            info="Select one or multiple"
        ),

        gr.File(
            label="📄 Upload Your Resume",
            file_types=[".pdf", ".docx", ".txt"],
            type="filepath"
        )
    ],

    outputs=gr.Markdown(),

    title="🤖 AI Job Finder Agent",

    description="Find internship opportunities using your skills or resume.",

    submit_btn="🔍 Find Jobs"
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d9bf17387ee6841efd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [58]:
jobs = [
    {
        "title": "Data Analyst Intern",
        "company": "ABC Technologies",
        "location": "Remote",
        "skills": ["Python", "SQL", "Pandas", "Excel"],
        "experience": "Fresher",
        "apply_url": "https://example.com/data-analyst-intern"
    },
    {
        "title": "Machine Learning Intern",
        "company": "AI Solutions",
        "location": "Noida",
        "skills": ["Python", "Machine Learning", "Scikit-learn"],
        "experience": "Fresher",
        "apply_url": "https://example.com/ml-intern"
    },
    {
        "title": "Python Developer Intern",
        "company": "Tech Labs",
        "location": "Delhi",
        "skills": ["Python", "Django", "SQL"],
        "experience": "Fresher",
        "apply_url": "https://example.com/python-intern"
    },
    {
        "title": "Data Science Intern",
        "company": "DataWorks",
        "location": "Remote",
        "skills": ["Python", "Machine Learning", "Pandas", "NumPy"],
        "experience": "Fresher",
        "apply_url": "https://example.com/data-science-intern"
    }
]

print("Jobs updated with Apply links!")

Jobs updated with Apply links!


In [59]:
def get_missing_skills(user_skills, job_skills):

    user_normalized = set(
        normalize_skill(skill)
        for skill in user_skills
    )

    missing = []

    for skill in job_skills:
        normalized = normalize_skill(skill)

        if normalized not in user_normalized:
            missing.append(skill)

    return missing


def explain_job_match(user_skills, job):

    matched = [
        skill for skill in user_skills
        if normalize_skill(skill) in [
            normalize_skill(s)
            for s in job["skills"]
        ]
    ]

    missing = get_missing_skills(
        user_skills,
        job["skills"]
    )

    return matched, missing


print("Job explanation system ready!")

Job explanation system ready!


In [60]:
test_job = jobs[1]

matched, missing = explain_job_match(
    ["Python", "ML"],
    test_job
)

print("Matched:", matched)
print("Missing:", missing)

Matched: ['Python', 'ML']
Missing: ['Scikit-learn']


In [61]:
def final_job_finder(role, skills, locations, resume):

    resume_skills = []

    # Resume se skills
    if resume is not None:
        resume_text = extract_resume_text(resume)
        resume_skills = extract_skills_from_resume(resume_text)

    # Manually entered skills
    manual_skills = [
        s.strip()
        for s in skills.split(",")
        if s.strip()
    ]

    # Combine skills
    all_skills = manual_skills + resume_skills

    all_skills = list(dict.fromkeys(
        s.lower() for s in all_skills
    ))

    if not all_skills:
        return "⚠️ Please enter skills or upload your resume."

    location_text = ", ".join(locations)

    # Search jobs
    results = search_jobs(
        user_skills=all_skills,
        role=role,
        location=location_text
    )

    if not results:
        return """
## 😕 No matching jobs found

Try adding more skills or selecting Remote.
"""

    output = "## 🤖 Recommended Jobs\n\n"

    # Resume skills
    if resume_skills:
        output += "**📄 Skills detected from resume:** "
        output += ", ".join(resume_skills)
        output += "\n\n"

    # Job results
    for i, job in enumerate(results, 1):

        missing_skills = get_missing_skills(
            all_skills,
            job["all_skills"]
        )

        if missing_skills:
            missing_text = ", ".join(missing_skills)
        else:
            missing_text = "None 🎉"

        matched_text = ", ".join(
            job["matched_skills"]
        )

        output += f"""
### {i}. 💼 {job['title']}

**🏢 Company:** {job['company']}
**📍 Location:** {job['location']}
**⭐ Match Score:** {job['match_score']}%
**🛠️ Matched Skills:** {matched_text}
**⚠️ Missing Skills:** {missing_text}

**💡 Why this job?**
This job matches your **{matched_text}** skills and fits your selected role/location preferences.

**🔗 [Apply Now]({job['apply_url']})**

---

"""

    return output

In [62]:
def search_jobs(user_skills, role=None, location=None):

    results = []

    normalized_user_skills = [
        normalize_skill(skill)
        for skill in user_skills
    ]

    for job in jobs:

        job_skills = [
            normalize_skill(skill)
            for skill in job["skills"]
        ]

        matched_skills = []

        for original_skill, normalized_skill in zip(
            user_skills,
            normalized_user_skills
        ):
            if normalized_skill in job_skills:
                matched_skills.append(original_skill)

        if not matched_skills:
            continue

        # Role matching
        role_match = True

        if role:
            role_words = [
                word.lower()
                for word in role.split()
                if len(word) > 2
            ]

            job_title = job["title"].lower()

            role_match = any(
                word in job_title
                for word in role_words
            )

        if not role_match:
            continue

        # Location matching
        location_match = True

        if location:
            preferred_locations = [
                x.strip().lower()
                for x in location.split(",")
                if x.strip()
            ]

            job_location = job["location"].lower()

            location_match = (
                job_location in preferred_locations
                or job_location == "remote"
            )

        if not location_match:
            continue

        # Match score
        score = (
            len(matched_skills)
            / len(job_skills)
        ) * 100

        results.append({
            "title": job["title"],
            "company": job["company"],
            "location": job["location"],
            "matched_skills": matched_skills,
            "all_skills": job["skills"],
            "match_score": round(score, 2),

            # NEW
            "apply_url": job["apply_url"]
        })

    results.sort(
        key=lambda x: x["match_score"],
        reverse=True
    )

    return results


print("✅ Search function updated with Apply links!")

✅ Search function updated with Apply links!


In [63]:
test = search_jobs(
    ["Python", "ML"],
    "Machine Learning Intern",
    "Noida"
)

print(test)

[{'title': 'Machine Learning Intern', 'company': 'AI Solutions', 'location': 'Noida', 'matched_skills': ['Python', 'ML'], 'all_skills': ['Python', 'Machine Learning', 'Scikit-learn'], 'match_score': 66.67, 'apply_url': 'https://example.com/ml-intern'}, {'title': 'Data Science Intern', 'company': 'DataWorks', 'location': 'Remote', 'matched_skills': ['Python', 'ML'], 'all_skills': ['Python', 'Machine Learning', 'Pandas', 'NumPy'], 'match_score': 50.0, 'apply_url': 'https://example.com/data-science-intern'}, {'title': 'Data Analyst Intern', 'company': 'ABC Technologies', 'location': 'Remote', 'matched_skills': ['Python'], 'all_skills': ['Python', 'SQL', 'Pandas', 'Excel'], 'match_score': 25.0, 'apply_url': 'https://example.com/data-analyst-intern'}]


In [64]:
app = gr.Interface(
    fn=final_job_finder,
    inputs=[
        gr.Textbox(
            label="💼 Job Role",
            placeholder="e.g. Machine Learning Intern"
        ),

        gr.Textbox(
            label="🛠️ Skills",
            placeholder="e.g. Python, ML, SQL"
        ),

        gr.CheckboxGroup(
            choices=locations,
            label="📍 Preferred Locations",
            info="Select one or multiple"
        ),

        gr.File(
            label="📄 Upload Your Resume",
            file_types=[".pdf", ".docx", ".txt"],
            type="filepath"
        )
    ],

    outputs=gr.Markdown(),

    title="🤖 AI Job Finder Agent",

    description="""
    Find internship opportunities using your skills or resume.
    Get matched skills, missing skills, match score and Apply Now links.
    """,

    submit_btn="🔍 Find Jobs"
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://59c89548a9e6d1e444.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [65]:
role_choices = [
    "Data Analyst Intern",
    "Machine Learning Intern",
    "Data Science Intern",
    "Python Developer Intern",
    "Software Developer Intern"
]

print("✅ Role filter ready!")

✅ Role filter ready!


In [67]:
experience_choices = [
    "Fresher",
    "0-1 Years",
    "1-2 Years",
    "2+ Years"
]

print("✅ Experience filter ready!")

✅ Experience filter ready!


In [84]:
def search_jobs(
    user_skills,
    role=None,
    location=None,
    experience=None
):

    results = []

    # Normalize and remove duplicate skills
    normalized_user_skills = list(dict.fromkeys(
        normalize_skill(skill)
        for skill in user_skills
    ))

    for job in jobs:

        # Normalize job skills
        job_skills = list(dict.fromkeys(
            normalize_skill(skill)
            for skill in job["skills"]
        ))

        # Find matching skills
        matched_normalized = set(
            normalized_user_skills
        ) & set(job_skills)

        if not matched_normalized:
            continue

        # Role matching
        if role:
            role_words = [
                word.lower()
                for word in role.split()
                if len(word) > 2
            ]

            job_title = job["title"].lower()

            if not any(
                word in job_title
                for word in role_words
            ):
                continue

        # Location matching
        if location:

            preferred_locations = [
                x.strip().lower()
                for x in location.split(",")
                if x.strip()
            ]

            job_location = job["location"].lower()

            if not (
                job_location in preferred_locations
                or job_location == "remote"
            ):
                continue

        # Experience matching
        if experience:

            if job["experience"].lower() != experience.lower():
                continue

        # Convert normalized matches back to readable skills
        matched_skills = [
            skill for skill in user_skills
            if normalize_skill(skill) in matched_normalized
        ]

        # Remove duplicates like ML + Machine Learning
        matched_skills = list(dict.fromkeys(
            normalize_skill(skill)
            for skill in matched_skills
        ))

        # Correct score
        score = (
            len(matched_normalized)
            / len(job_skills)
        ) * 100

        results.append({
            "title": job["title"],
            "company": job["company"],
            "location": job["location"],
            "matched_skills": matched_skills,
            "all_skills": job["skills"],
            "match_score": round(min(score, 100), 2),
            "experience": job["experience"],
            "apply_url": job["apply_url"]
        })

    results.sort(
        key=lambda x: x["match_score"],
        reverse=True
    )

    return results[:10]


print("✅ Match score fixed!")

✅ Match score fixed!


In [79]:
test = search_jobs(
    user_skills=["Python", "ML"],
    role="Machine Learning Intern",
    location="Noida",
    experience="Fresher"
)

print(test)

[{'title': 'Machine Learning Intern', 'company': 'AI Solutions', 'location': 'Noida', 'matched_skills': ['Python', 'ML'], 'all_skills': ['Python', 'Machine Learning', 'Scikit-learn'], 'match_score': 66.67, 'experience': 'Fresher', 'apply_url': 'https://example.com/ml-intern'}, {'title': 'Data Science Intern', 'company': 'DataWorks', 'location': 'Remote', 'matched_skills': ['Python', 'ML'], 'all_skills': ['Python', 'Machine Learning', 'Pandas', 'NumPy'], 'match_score': 50.0, 'experience': 'Fresher', 'apply_url': 'https://example.com/data-science'}, {'title': 'Data Analyst Intern', 'company': 'ABC Technologies', 'location': 'Remote', 'matched_skills': ['Python'], 'all_skills': ['Python', 'SQL', 'Pandas', 'Excel'], 'match_score': 25.0, 'experience': 'Fresher', 'apply_url': 'https://example.com/data-analyst'}]


In [80]:
app = gr.Interface(
    fn=final_job_finder,

    inputs=[
        gr.Dropdown(
            choices=role_choices,
            label="💼 Job Role",
            value="Data Science Intern"
        ),

        gr.Textbox(
            label="🛠️ Skills",
            placeholder="e.g. Python, ML, SQL"
        ),

        gr.CheckboxGroup(
            choices=locations,
            label="📍 Preferred Locations",
            info="Select one or multiple"
        ),

        gr.File(
            label="📄 Upload Your Resume",
            file_types=[".pdf", ".docx", ".txt"],
            type="filepath"
        )
    ],

    outputs=gr.Markdown(),

    title="🤖 AI Job Finder Agent",

    description="""
    Find internship opportunities using your skills or resume.

    Get:
    ⭐ Match Score
    🛠️ Matched Skills
    ⚠️ Missing Skills
    💡 Why this job?
    🔗 Apply Now
    """,

    submit_btn="🔍 Find Jobs"
)

app.launch(share=True)

/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1281: UserWarning: Expected 5 arguments for function <function final_job_finder at 0x7a7fb5a776a0>, received 4.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/gradio/utils.py:1285: UserWarning: Expected at least 5 arguments for function <function final_job_finder at 0x7a7fb5a776a0>, received 4.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3764dd52fc2dd6bb6c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [71]:
def final_job_finder(role, skills, locations, experience, resume):

    resume_skills = []

    # Resume se skills
    if resume is not None:
        resume_text = extract_resume_text(resume)
        resume_skills = extract_skills_from_resume(resume_text)

    # Manual skills
    manual_skills = [
        s.strip()
        for s in skills.split(",")
        if s.strip()
    ]

    # Combine skills
    all_skills = manual_skills + resume_skills

    all_skills = list(dict.fromkeys(
        s.lower() for s in all_skills
    ))

    if not all_skills:
        return "⚠️ Please enter skills or upload your resume."

    # Locations
    location_text = ", ".join(locations)

    # Search
    results = search_jobs(
        user_skills=all_skills,
        role=role,
        location=location_text,
        experience=experience
    )

    if not results:
        return """
## 😕 No matching jobs found

Try:
- adding more skills
- selecting Remote
- changing the job role
- changing the experience level
"""

    output = "## 🤖 Recommended Jobs\n\n"

    # Resume skills
    if resume_skills:
        output += "**📄 Skills detected from resume:** "
        output += ", ".join(resume_skills)
        output += "\n\n"

    # Results
    for i, job in enumerate(results, 1):

        missing_skills = get_missing_skills(
            all_skills,
            job["all_skills"]
        )

        if missing_skills:
            missing_text = ", ".join(missing_skills)
        else:
            missing_text = "None 🎉"

        matched_text = ", ".join(
            job["matched_skills"]
        )

        output += f"""
### {i}. 💼 {job['title']}

**🏢 Company:** {job['company']}
**📍 Location:** {job['location']}
**💼 Experience:** {job['experience']}
**⭐ Match Score:** {job['match_score']}%
**🛠️ Matched Skills:** {matched_text}
**⚠️ Missing Skills:** {missing_text}

**💡 Why this job?**
This job matches your **{matched_text}** skills and fits your selected role, location and experience preferences.

**🔗 [Apply Now]({job['apply_url']})**

---

"""

    return output


print("✅ Final job finder updated!")

✅ Final job finder updated!


In [73]:
app = gr.Interface(
    fn=final_job_finder,

    inputs=[
        gr.Dropdown(
            choices=role_choices,
            label="💼 Job Role",
            value="Data Science Intern"
        ),

        gr.Textbox(
            label="🛠️ Skills",
            placeholder="e.g. Python, ML, SQL"
        ),

        gr.CheckboxGroup(
            choices=locations,
            label="📍 Preferred Locations",
            info="Select one or multiple"
        ),

        gr.Dropdown(
            choices=experience_choices,
            label="💼 Experience",
            value="Fresher"
        ),

        gr.File(
            label="📄 Upload Your Resume",
            file_types=[".pdf", ".docx", ".txt"],
            type="filepath"
        )
    ],

    outputs=gr.Markdown(),

    title="🤖 AI Job Finder Agent",

    description="""
    Find internship opportunities using your skills or resume.

    ⭐ Match Score
    🛠️ Matched Skills
    ⚠️ Missing Skills
    💡 Why this job?
    🔗 Apply Now
    """,

    submit_btn="🔍 Find Jobs"
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b9ab503809882bffb1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [81]:
jobs = [
    {
        "title": "Data Analyst Intern",
        "company": "ABC Technologies",
        "location": "Remote",
        "skills": ["Python", "SQL", "Pandas", "Excel"],
        "experience": "Fresher",
        "apply_url": "https://example.com/data-analyst"
    },
    {
        "title": "Machine Learning Intern",
        "company": "AI Solutions",
        "location": "Noida",
        "skills": ["Python", "Machine Learning", "Scikit-learn"],
        "experience": "Fresher",
        "apply_url": "https://example.com/ml-intern"
    },
    {
        "title": "Python Developer Intern",
        "company": "Tech Labs",
        "location": "Delhi",
        "skills": ["Python", "Django", "SQL"],
        "experience": "Fresher",
        "apply_url": "https://example.com/python-intern"
    },
    {
        "title": "Data Science Intern",
        "company": "DataWorks",
        "location": "Remote",
        "skills": ["Python", "Machine Learning", "Pandas", "NumPy"],
        "experience": "Fresher",
        "apply_url": "https://example.com/data-science"
    },
    {
        "title": "AI Intern",
        "company": "Future AI",
        "location": "Bangalore",
        "skills": ["Python", "Machine Learning", "Deep Learning"],
        "experience": "Fresher",
        "apply_url": "https://example.com/ai-intern"
    },
    {
        "title": "Business Data Analyst Intern",
        "company": "DataCorp",
        "location": "Mumbai",
        "skills": ["Python", "SQL", "Excel", "Power BI"],
        "experience": "Fresher",
        "apply_url": "https://example.com/business-analyst"
    },
    {
        "title": "Junior Data Scientist Intern",
        "company": "Analytics Hub",
        "location": "Pune",
        "skills": ["Python", "Pandas", "NumPy", "Machine Learning"],
        "experience": "Fresher",
        "apply_url": "https://example.com/junior-ds"
    },
    {
        "title": "ML Engineer Intern",
        "company": "NeuralTech",
        "location": "Gurgaon",
        "skills": ["Python", "Machine Learning", "Scikit-learn", "SQL"],
        "experience": "Fresher",
        "apply_url": "https://example.com/ml-engineer"
    },
    {
        "title": "Python Data Intern",
        "company": "CloudData",
        "location": "Hyderabad",
        "skills": ["Python", "SQL", "Pandas"],
        "experience": "Fresher",
        "apply_url": "https://example.com/python-data"
    },
    {
        "title": "Data Science Trainee",
        "company": "Insight Analytics",
        "location": "Kolkata",
        "skills": ["Python", "Machine Learning", "SQL", "Pandas"],
        "experience": "Fresher",
        "apply_url": "https://example.com/data-trainee"
    }
]

print("✅ 10 demo jobs loaded!")

✅ 10 demo jobs loaded!


In [82]:
test = search_jobs(
    user_skills=["Python"],
    role=None,
    location=None,
    experience="Fresher"
)

print("Total jobs found:", len(test))

for job in test:
    print(job["title"], "-", job["company"])

Total jobs found: 10
Machine Learning Intern - AI Solutions
Python Developer Intern - Tech Labs
AI Intern - Future AI
Python Data Intern - CloudData
Data Analyst Intern - ABC Technologies
Data Science Intern - DataWorks
Business Data Analyst Intern - DataCorp
Junior Data Scientist Intern - Analytics Hub
ML Engineer Intern - NeuralTech
Data Science Trainee - Insight Analytics


In [85]:
app = gr.Interface(
    fn=final_job_finder,

    inputs=[
        gr.Dropdown(
            choices=role_choices,
            label="💼 Job Role",
            value=None,
            allow_custom_value=True
        ),

        gr.Textbox(
            label="🛠️ Skills",
            placeholder="e.g. Python, ML, SQL"
        ),

        gr.CheckboxGroup(
            choices=locations,
            label="📍 Preferred Locations",
            info="Select one or multiple"
        ),

        gr.Dropdown(
            choices=experience_choices,
            label="💼 Experience",
            value="Fresher"
        ),

        gr.File(
            label="📄 Upload Your Resume",
            file_types=[".pdf", ".docx", ".txt"],
            type="filepath"
        )
    ],

    outputs=gr.Markdown(),

    title="🤖 AI Job Finder Agent",

    description="""
    Find internship opportunities using your skills or resume.

    ⭐ Match Score
    🛠️ Matched Skills
    ⚠️ Missing Skills
    💡 Why this job?
    🔗 Apply Now
    """,

    submit_btn="🔍 Find Jobs"
)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://788b2d671b1bc1cffc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
